# Importing microbiome data

**R equivalent:** `import_biom()`, `import_qiime()`, `import_mothur()`, manual `phyloseq()` construction

This notebook shows how to load data from every format pyloseq supports and
construct a `Phyloseq` object.

In [3]:
import pandas as pd

from pyloseq import (
    OtuTable,
    Phyloseq,
    PhyTree,
    SampleData,
    TaxTable,
)
from pyloseq.datasets.fixtures import (
    load_global_patterns_reference,
)


## Manual construction from DataFrames

The most flexible approach — useful when data is already in Python.

In [4]:
# OTU table: taxa as rows, samples as columns
otu_data = pd.DataFrame(
    {"Sample1": [100, 50, 0, 20],
     "Sample2": [80, 30, 10, 0],
     "Sample3": [0, 60, 40, 5]},
    index=["OTU1", "OTU2", "OTU3", "OTU4"],
)
otu = OtuTable(otu_data, taxa_are_rows=True)

# Sample metadata
sam = SampleData(pd.DataFrame(
    {"SampleType": ["Gut", "Skin", "Oral"],
     "Subject":   ["A",   "A",    "B"]},
    index=["Sample1", "Sample2", "Sample3"],
))

# Taxonomic classification
tax = TaxTable(pd.DataFrame(
    {"Phylum":  ["Firmicutes", "Bacteroidetes", "Proteobacteria", "Firmicutes"],
     "Genus":   ["Lactobacillus", "Bacteroides", "Pseudomonas", "Clostridium"]},
    index=["OTU1", "OTU2", "OTU3", "OTU4"],
))

ps = Phyloseq(otu=otu, sam=sam, tax=tax)
print(ps)

Phyloseq(4 taxa × 3 samples)
  sample_data: 2 variables
  tax_table:   2 ranks


## From the golden reference datasets

These mirror R's `data("GlobalPatterns")` etc.

In [5]:
ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
    tree=PhyTree.from_newick(ref["phy_tree_newick"]),
)
print(gp)
print(f"\ntaxa: {gp.ntaxa}, samples: {gp.nsamples}")

Phyloseq(19216 taxa × 26 samples)
  sample_data: 7 variables
  tax_table:   7 ranks
  phy_tree:    19216 tips

taxa: 19216, samples: 26


In [6]:
# R reference: ntaxa(GlobalPatterns) == 19216, nsamples == 26
assert gp.ntaxa == 19216, f"Expected 19216 taxa, got {gp.ntaxa}"
assert gp.nsamples == 26, f"Expected 26 samples, got {gp.nsamples}"
print("✓ GlobalPatterns dimensions match R reference")

✓ GlobalPatterns dimensions match R reference


## Inspect the object

**R equivalent:** `taxa_names()`, `sample_names()`, `rank_names()`, `sample_variables()`

In [7]:
print("Rank names:      ", gp.rank_names)
print("Sample variables:", gp.sample_variables)
print("Sample names[:3]:", list(gp.sample_names[:3]))
print("Taxa names[:3]:  ", list(gp.taxa_names[:3]))

Rank names:       ['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species']
Sample variables: ['X.SampleID', 'Primer', 'Final_Barcode', 'Barcode_truncated_plus_T', 'Barcode_full_length', 'SampleType', 'Description']
Sample names[:3]: ['CL3', 'CC1', 'SV1']
Taxa names[:3]:   ['549322', '522457', '951']
